# Cercha 11.2: construya el cálculo

Mismo ejemplo 11.2, láminas217–225. No asigne un material ficticio: E_ref se usa solo para comprobar coeficientes.

Primero resuelva a mano. Complete cada `None` con una expresión o matriz. Ejecute una celda cada vez. Los `assert` comprueban el procedimiento, no asignan calificaciones. Una celda incompleta se detiene con una indicación. Python es complementario.

Use kN, m y rad, excepto donde se indique una conversión. +X a la derecha, +Y arriba, giros y momentos antihorarios. $S_{ij}>0$ indica tensión. $F_n$ es carga libre; $F_a$ es acción neta en apoyos. La reacción es $F_a$ menos la carga aplicada allí. $w>0$ actúa hacia abajo en la viga horizontal.

Referencia: https://view.genially.com/684f80b53c7df3efd8dc19f4


In [ ]:
import numpy as np
np.set_printoptions(precision=7, suppress=True)
def complete(value, shape, name):
    assert value is not None, f"Complete {name} antes de ejecutar la comprobación."
    out=np.asarray(value,dtype=float)
    assert out.shape==shape, f"Revise dimensiones de {name}: se espera {shape}."
    assert np.isfinite(out).all(), f"Revise valores finitos de {name}."
    return out


## 1. Calcule la geometría

Nudos 1=(2,0), 2=(0,0), 3=(0,2) m. Barras 1→2, 1→3 y 2→3. En Python los índices empiezan en 0. Forme una fila [L,c,s] por barra.


In [ ]:
# Actividad C1
xy=np.array([[2.,0.],[0.,0.],[0.,2.]])
conectividad=[(0,1),(0,2),(1,2)]
geometria=None


In [ ]:
# Compruebe el paso antes de continuar.
geometria=complete(geometria,(3,3),'geometria')
assert np.allclose(geometria[:,0],[2,2*np.sqrt(2),2]), 'Revise las longitudes.'
assert np.allclose(geometria[:,1:], [[-1,0],[-1/np.sqrt(2),1/np.sqrt(2)],[0,1]]), 'Revise el sentido i→j y el cuadrante.'
print(geometria)


## 2. Calcule las matrices de barra

A/L=1 mm=0.001 m en cada barra. Use E_ref=1 kN/mm²=10⁶ kN/m² como normalización algebraica, no como material real. Complete Kbarra(c,s) con la matriz global 4×4.


In [ ]:
# Actividad C2
E=1e6
def Kbarra(c,s):
    return None
matrices=[Kbarra(c,s) for L,c,s in geometria]


In [ ]:
# Compruebe el paso antes de continuar.
matrices=[complete(k,(4,4),'Kbarra') for k in matrices]
for k in matrices:
    assert np.allclose(k,k.T), 'La matriz de barra debe ser simétrica.'
    assert np.linalg.matrix_rank(k)==1, 'Una barra axial aislada aporta un solo modo resistente.'
assert np.isclose(matrices[1][0,1],-500), 'Revise el término cs de la diagonal.'
print(matrices[1])


## 3. Ensamble y particione

Orden global [u1,v1,u2,v2,u3,v3]. Son libres [u1,v1,v2]. Forme K y extraiga Knn con el mismo orden.


In [ ]:
# Actividad C3
codigos=[[0,1,2,3],[0,1,4,5],[2,3,4,5]]
K=None
n=np.array([0,1,3]); a=np.array([2,4,5])
Knn=None


In [ ]:
# Compruebe el paso antes de continuar.
K=complete(K,(6,6),'K');Knn=complete(Knn,(3,3),'Knn')
assert np.allclose(Knn,[[1500,-500,0],[-500,500,0],[0,0,1000]]), 'Revise códigos, sumas y extracción del bloque.'
assert np.allclose(Knn,K[np.ix_(n,n)]), 'Knn debe extraerse de K en el orden de n.'
print(Knn)


## 4. Resuelva y recupere fuerzas

Fn=[40,−50,0] kN. No hay fuerzas de empotramiento. Los apoyos son inmóviles y no reciben cargas nodales directas. Calcule δn, Fa y Sij.


In [ ]:
# Actividad C4
Fn=np.array([40.,-50.,0.])
dn=None
delta=np.zeros(6)
Fa=None
S=None


In [ ]:
# Compruebe el paso antes de continuar.
dn=complete(dn,(3,),'dn');Fa=complete(Fa,(3,),'Fa');S=complete(S,(3,),'S')
assert np.allclose(dn,[-.01,-.11,0]), 'Revise la solución y las unidades m/mm.'
assert np.allclose(Fa,[10,-50,50]), 'Revise las filas de apoyo.'
assert np.allclose(S,[-10,50*np.sqrt(2),0]), 'Revise proyección y tensión positiva.'
assert np.allclose(Knn@dn,Fn), 'Compruebe sustitución en las tres ecuaciones.'
print('dn (m):',dn,'; Fa (kN):',Fa,'; S (kN):',S)


## Explique el resultado

¿Por qué v2 se desacopla? Escriba las dos ecuaciones del bloque [u1,v1]. Compruebe el equilibrio del nudo1: las componentes de S12 y S13 deben equilibrar 40 kN y −50 kN. Explique por qué la barra23 no tiene fuerza.

Si una prueba falla, compare la primera operación distinta, no solo la respuesta final. Consulte `soluciones_laboratorios.md` después de intentar el paso. El notebook resuelto del mismo ejemplo sirve como referencia adicional.
